# MLPerf Inference — BERT-Large / SQuAD v1.1 (local, WSL `mlperf` distro)

The NLP question-answering benchmark, run on your RTX 5070 Ti. Expected **f1 ≈ 90.4** on 1000 examples.

> **How to run (in the `mlperf` WSL distro):**
> ```bash
> wsl -d mlperf
> source /root/mlperf/venv/bin/activate
> pip install -q jupyterlab ipykernel
> python -m ipykernel install --user --name mlperf --display-name 'mlperf (venv)'
> jupyter lab --no-browser --ip 0.0.0.0 --port 8888   # open the printed URL
> ```
> Then open this notebook, pick the **mlperf (venv)** kernel, and Run All.
> Assets are cached under `/root/mlperf/…`, so re-runs skip downloads. Works on a fresh
> distro too (it clones + downloads on first run). Needs torch 2.11+cu128 already in the venv.

## 1 · GPU check

In [1]:
import torch
print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

torch 2.11.0+cu128 | cuda True | NVIDIA GeForce RTX 5070 Ti Laptop GPU


## 2 · Deps (torch cu128 already in venv)

In [ ]:
%pip install -q "mlcommons-loadgen==6.0.16" "transformers==4.48.3"
import mlperf_loadgen, transformers; print('loadgen + transformers',transformers.__version__)

## 3 · Repo + model + dataset (cached under /root/mlperf)

In [ ]:
%%bash
set -euo pipefail
INFERENCE_REF=da738a5     # pin the harness (mlcommons/inference is a moving master)
verify(){ echo "$1  $2" | sha256sum -c - >/dev/null || { echo "!! checksum mismatch: $2"; exit 1; }; }

if [ ! -d /root/mlperf/inference ]; then
  git clone --filter=blob:none --no-checkout https://github.com/mlcommons/inference.git /root/mlperf/inference
fi
git -C /root/mlperf/inference fetch --filter=blob:none -q origin "$INFERENCE_REF" 2>/dev/null || true
W=$(git -C /root/mlperf/inference rev-parse -q --verify "$INFERENCE_REF^{commit}")
git -C /root/mlperf/inference reset --hard -q "$W"    # pin exactly, every run

D=/root/mlperf/inference/language/bert/build/data/bert_tf_v1_1_large_fp32_384_v2
mkdir -p "$D" /root/mlperf/inference/language/bert/build/logs
DJ=/root/mlperf/inference/language/bert/build/data/dev-v1.1.json

# each asset is downloaded then SHA-256-verified before use (fail closed on tamper/corruption)
[ -s "$DJ" ] || wget -q -O "$DJ" https://raw.githubusercontent.com/rajpurkar/SQuAD-explorer/master/dataset/dev-v1.1.json
verify 95aa6a52d5d6a735563366753ca50492a658031da74f301ac5238b03966972c9 "$DJ"
[ -s "$D/vocab.txt" ] || wget -q -O "$D/vocab.txt" 'https://zenodo.org/record/3733896/files/vocab.txt?download=1'
verify 07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3 "$D/vocab.txt"
[ -s "$D/model.pytorch" ] || wget -q -O "$D/model.pytorch" 'https://zenodo.org/record/3733896/files/model.pytorch?download=1'
verify 71af14acc3cb47ebd88e028d9ff8a5f06e15f6ed666e16293b7ad2539171397f "$D/model.pytorch"
echo "verified:"; ls -lh "$D/model.pytorch"

## 4 · Self-contained tokenizer + Blackwell math-SDP guard

`sitecustomize.py` forces the math SDP backend (required on sm_120, harmless elsewhere).

In [4]:
%%writefile /root/mlperf/inference/language/bert/tokenization.py
import unicodedata
def convert_to_unicode(t):
    return t if isinstance(t,str) else t.decode('utf-8','ignore')
def printable_text(t):
    return t if isinstance(t,str) else t.decode('utf-8','ignore')
def whitespace_tokenize(t):
    t=t.strip(); return t.split() if t else []
class BasicTokenizer(object):
    def __init__(self, do_lower_case=True): self.do_lower_case=do_lower_case
    def tokenize(self, text):
        text=self._clean(convert_to_unicode(text)); out=[]
        for tok in whitespace_tokenize(text):
            if self.do_lower_case: tok=self._strip(tok.lower())
            out.extend(self._punc(tok))
        return whitespace_tokenize(' '.join(out))
    def _strip(self,t):
        return ''.join(c for c in unicodedata.normalize('NFD',t) if unicodedata.category(c)!='Mn')
    def _punc(self,t):
        r=[]; nw=True
        for c in t:
            if _punct(c): r.append([c]); nw=True
            else:
                if nw: r.append([])
                nw=False; r[-1].append(c)
        return [''.join(x) for x in r]
    def _clean(self,t):
        o=[]
        for c in t:
            cp=ord(c)
            if cp==0 or cp==0xFFFD or _ctrl(c): continue
            o.append(' ' if _ws(c) else c)
        return ''.join(o)
def _ws(c):
    return c in (' ','\t','\n','\r') or unicodedata.category(c)=='Zs'
def _ctrl(c):
    return False if c in ('\t','\n','\r') else unicodedata.category(c) in ('Cc','Cf')
def _punct(c):
    cp=ord(c)
    if (33<=cp<=47) or (58<=cp<=64) or (91<=cp<=96) or (123<=cp<=126): return True
    return unicodedata.category(c).startswith('P')

Overwriting /root/mlperf/inference/language/bert/tokenization.py


In [5]:
%%writefile /root/mlperf/inference/language/bert/sitecustomize.py
try:
    import torch
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    print('[sitecustomize] math SDP forced')
except Exception as e: print('SDP guard skipped:', e)

Overwriting /root/mlperf/inference/language/bert/sitecustomize.py


## 5 · Performance (Offline, bounded)

In [6]:
%%bash
source /root/mlperf/venv/bin/activate
cd /root/mlperf/inference/language/bert
cat > user_demo.conf <<'CONF'
*.Offline.target_qps = 40.0
*.Offline.min_duration = 60000
*.Offline.min_query_count = 1
CONF
python run.py --backend=pytorch --scenario=Offline --user_conf=user_demo.conf --max_examples=1000 2>&1 | tail -3
echo '----- SUMMARY -----'; sed -n '1,16p' build/logs/mlperf_log_summary.txt

For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

----- SUMMARY -----


## 6 · Accuracy + f1

`run.py --accuracy` auto-scores with wrong default paths (harmless) → score explicitly after.

In [7]:
%%bash
source /root/mlperf/venv/bin/activate
cd /root/mlperf/inference/language/bert
python run.py --backend=pytorch --scenario=Offline --accuracy --max_examples=1000 2>&1 | tail -2 || true
echo '----- f1 -----'
python accuracy-squad.py --vocab_file build/data/bert_tf_v1_1_large_fp32_384_v2/vocab.txt \
  --val_data build/data/dev-v1.1.json --log_file build/logs/mlperf_log_accuracy.json \
  --out_file build/logs/predictions.json --max_examples 1000 2>&1 | grep -Ei '"f1"|exact_match' | tail -1

    raise CalledProcessError(retcode, cmd)
subprocess.CalledProcessError: Command 'python3 /root/mlperf/inference/language/bert/accuracy-squad.py --max_examples 1000' returned non-zero exit status 1.
----- f1 -----
{"exact_match": 86.6, "f1": 90.40153169843566}


## Done ✅ — BERT-Large / SQuAD on your GPU.